# 10 · 입력 폴더 정리

**목적:** 압축을 푼 데이터 상위 폴더 하나를 지정하면, 00이 읽는 `input_data` 구조로 모은다. 실행 순서는 **10 → 00 → 01 → … → 09**다.

```text
지정한 원본 폴더/                       이 노트북의 폴더/
  Training/묶음A/annotation_person/       input_data/annotation_person/
  Validation/묶음B/labels/                input_data/labels/
  다른묶음/emr/                           input_data/emr/
```

중간 폴더 깊이는 상관없다. JSON의 **바로 위 폴더 이름**이 `annotation_person`, `labels`, `emr`인 파일을 찾는다. 폴더 이름은 대소문자를 구분하고 확장자는 `.json`·`.JSON` 모두 읽는다. 압축 해제와 이미지 복사는 하지 않는다. 하위 심볼릭 링크도 따라가지 않는다.

원본 파일은 수정하지 않는다. 같은 종류·ID의 중복을 정리하고, 세 종류가 모두 있는 기록만 복사한다. 파형·라벨·임상값의 유효성 판정은 기존 00이 수행한다.

## 1. 상위 폴더를 지정한다

Jupyter 작업 디렉터리를 `04-더-요약된-파일`로 두고 실행한다. 현재 설정은 `../../dataset`이다. 이는 작업 디렉터리에서 두 단계 위의 `dataset`을 가리킨다. 다른 위치를 사용하려면 아래 `SOURCE`만 바꾼다. 원본 폴더는 미리 압축을 풀어 둔다.

출력은 집계와 상태만 보여준다. 파일별 원본 경로·ID·해시는 내부 `data/input_import/실행번호/`에 저장하므로, 이 폴더는 반출용 요약이 아니다. 코드에 적은 경로도 노트북에 저장되므로 반출할 때 확인한다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import errno
import json
import os
import shutil
import uuid

import pandas as pd
from IPython.display import display

WORKSPACE = Path.cwd()
SOURCE = Path("../../dataset")  # Change this directory to select another source.
TARGET = WORKSPACE / "input_data"
REPORTS = WORKSPACE / "data" / "input_import"
KINDS = ("annotation_person", "labels", "emr")


In [ ]:
# Relative paths start at the notebook kernel's working directory.
# Example: ../../dataset means two directories above WORKSPACE, then dataset.
setup_code = "ready"
try:
    SOURCE = SOURCE.expanduser().resolve(strict=True)
except (OSError, RuntimeError, ValueError):
    setup_code = "source_unavailable"
print("입력 상태:", setup_code)
if setup_code == "source_unavailable":
    print("SOURCE 폴더를 찾거나 읽을 수 없습니다. 상대경로의 기준과 압축 해제 여부를 확인하세요.")


In [ ]:
def check_locations(source, target, reports):
    if not (WORKSPACE / "00-데이터-준비.ipynb").is_file():
        raise ValueError("wrong_working_directory")
    if not source.is_dir():
        raise ValueError("source_not_directory")
    if target.is_symlink() or (target.exists() and not target.is_dir()):
        raise ValueError("target_not_regular_directory")
    # Never move the source itself into a backup or scan prior generated results.
    if source == target.resolve() or target.resolve() in source.parents:
        raise ValueError("source_inside_target")
    data_root = (WORKSPACE / "data").resolve()
    if source == data_root or data_root in source.parents:
        raise ValueError("source_inside_generated_data")
    # Staging and backups must stay inside this workspace's real data directory.
    if (WORKSPACE / "data").is_symlink() or reports.is_symlink():
        raise ValueError("report_directory_is_symlink")

In [ ]:
RUN = None
issues = []
if setup_code == "ready":
    try:
        check_locations(SOURCE, TARGET, REPORTS)
        run_name = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]
        candidate_run = REPORTS / run_name
        candidate_run.mkdir(parents=True, exist_ok=False)
        RUN = candidate_run
    except (OSError, ValueError) as error:
        # Keep detailed paths out of notebook outputs, including OS exception text.
        setup_code = str(error) if isinstance(error, ValueError) else "setup_io_error"
print("준비 상태:", setup_code)

## 2. 하위 폴더에서 후보 파일을 찾는다

02의 재귀 탐색 아이디어를 사용하되, 탐색 오류를 놓치지 않도록 `os.walk`의 오류도 기록한다. 지정한 원본이 작업 폴더의 상위에 있더라도 현재 `input_data`와 `data`는 탐색에서 제외한다. 이전 입력이나 백업을 새 원본으로 다시 섞지 않기 위해서다.

In [ ]:
def source_files(source, excluded, issues, scan_counts):
    def walk_error(error):
        # An unreadable directory is logged; os.walk continues with its siblings.
        issues.append(dict(code="scan_io_error", kind="", record_id="",
                           path=str(error.filename or ""), error_type=type(error).__name__))

    for folder, directories, filenames in os.walk(source, onerror=walk_error, followlinks=False):
        folder = Path(folder)
        kept = []
        for name in sorted(directories):
            child = folder / name
            try:
                if child.is_symlink():
                    scan_counts["skipped_links"] += 1
                elif child.resolve() not in excluded:
                    kept.append(name)
            except OSError as error:
                walk_error(error)
        directories[:] = kept
        # Example: source/A/labels/DEMO684219_0.JSON -> kind labels, same record stem.
        if folder.name not in KINDS:
            continue
        for name in sorted(filenames):
            path = folder / name
            if path.suffix.lower() != ".json":
                continue
            if path.is_symlink():
                scan_counts["skipped_links"] += 1
                continue
            yield folder.name, path


In [ ]:
def read_candidate(path, kind):
    content = path.read_bytes()
    document = json.loads(content.decode("utf-8-sig"))
    if not isinstance(document, dict):
        raise ValueError("json_not_object")
    # 00 uses the filename stem to join labels and EMR; preserve it exactly.
    # Example: file DEMO684219_0.json must contain ID="DEMO684219_0".
    if kind in ("labels", "emr") and str(document.get("ID")) != path.stem:
        raise ValueError("internal_id_mismatch")
    entry = dict(path=str(path), sha256=hashlib.sha256(content).hexdigest(), size=len(content))
    return entry, document

## 3. 중복을 판정한다

같은 종류와 파일명 stem을 같은 기록으로 본다. 바이트가 같으면 중복 하나로 합친다. 바이트가 달라도 JSON 내용이 같으면 공백·키 순서 차이로 보고 합친다.

`labels`는 `Bbox`를 제외한 내용이 같고 한쪽에만 비어 있지 않은 `Bbox`가 있으면 그 파일을 선택한다. 두 파일에 서로 다른 Bbox가 있거나 다른 필드가 다르면 충돌이다. 충돌·JSON 읽기 오류·내부 ID 불일치는 **해당 ID의 기록 전체를 제외**하고 다음 기록으로 넘어간다. 나중에 정상 중복이 나타나도 제외를 취소하지 않는다. 서로 다른 내용을 임의로 선택하지 않기 위해서다.


In [ ]:
def duplicate_choice(kind, previous, candidate):
    if previous == candidate:
        return "keep_previous"
    if kind == "labels":
        # Copy dictionaries before removing Bbox; never alter parsed source documents.
        left, right = dict(previous), dict(candidate)
        left_boxes = left.pop("Bbox", None)
        right_boxes = right.pop("Bbox", None)
        if left == right and (left_boxes == right_boxes or not left_boxes or not right_boxes):
            # Example: no boxes vs one box -> prefer the file containing that box.
            if right_boxes and not left_boxes:
                return "use_candidate"
            return "keep_previous"
    return "conflict"

In [ ]:
# Fictional documents only; no real record or source path is displayed.
example_plain = {"ID": "DEMO684219_0", "Abnormality": "0_1"}
example_boxed = dict(example_plain, Bbox=[[100, 0, 200, 80, 200, 80]])
example_conflict = dict(example_plain, Abnormality="1_1")
print("same content =", duplicate_choice("labels", example_plain, dict(example_plain)))
print("with Bbox =", duplicate_choice("labels", example_plain, example_boxed))
print("different labels =", duplicate_choice("labels", example_plain, example_conflict))

In [ ]:
def add_candidate(catalog, kind, path, inventory):
    entry, document = read_candidate(path, kind)
    record_id = path.stem
    previous = catalog[kind].get(record_id)
    decision = "first_copy"
    if previous is not None:
        decision = "keep_previous"
        if previous["sha256"] != entry["sha256"]:
            old_entry, old_document = read_candidate(Path(previous["path"]), kind)
            if old_entry["sha256"] != previous["sha256"]:
                raise ValueError("source_changed_during_scan")
            decision = duplicate_choice(kind, old_document, document)
    # Store every candidate for the internal audit, including conflicting duplicates.
    inventory.append(dict(kind=kind, record_id=record_id, decision=decision, **entry))
    if decision in ("first_copy", "use_candidate"):
        catalog[kind][record_id] = entry
    if decision == "conflict":
        raise ValueError("conflicting_duplicate")

In [ ]:
catalog = {kind: {} for kind in KINDS}
inventory = []
invalid_records = set()
scan_counts = dict(skipped_links=0)
found_counts = {kind: 0 for kind in KINDS}
if RUN is not None:
    excluded = {TARGET.resolve(), (WORKSPACE / "data").resolve()}
    for kind, path in source_files(SOURCE, excluded, issues, scan_counts):
        found_counts[kind] += 1
        try:
            add_candidate(catalog, kind, path, inventory)
        except (OSError, ValueError, TypeError, UnicodeError, RecursionError) as error:
            # One invalid duplicate excludes the whole record, even if another copy is valid.
            # Example: conflicting labels for DEMO_A_0 do not exclude DEMO_B_0.
            invalid_records.add(path.stem)
            known = {
                "json_not_object", "internal_id_mismatch",
                "source_changed_during_scan", "conflicting_duplicate",
            }
            code = str(error) if str(error) in known else "unreadable_json"
            issues.append(dict(code=code, kind=kind, record_id=path.stem,
                               path=str(path), error_type=type(error).__name__))
print("검사한 JSON 수:", sum(found_counts.values()))
print("건너뛴 링크 수:", scan_counts["skipped_links"])
print("오류가 있는 기록 수:", len(invalid_records))


## 4. 세 JSON이 모두 있는 기록을 고른다

00은 같은 ID의 annotation·labels·emr를 함께 읽는다. 세 종류 중 하나라도 없으면 이번 복사에서 해당 기록을 제외하고 내부 제외 표에 남긴다. 예를 들어 annotation에 A·B, labels에 A·B, emr에 A만 있으면 A만 복사한다. 세 종류가 있더라도 앞 단계에서 오류가 발견된 ID는 제외한다.

이 단계는 파일의 조합을 맞추는 작업이다. 선택한 기록 안의 신호 길이·판독 개수·산모 ID 등은 00의 기존 검사와 제외 규칙을 따른다.


In [ ]:
def complete_records(catalog):
    groups = [set(catalog[kind]) for kind in KINDS]
    all_ids = set.union(*groups)
    complete = set.intersection(*groups)
    missing = []
    for record_id in sorted(all_ids - complete):
        absent = [kind for kind in KINDS if record_id not in catalog[kind]]
        missing.append(dict(record_id=record_id, missing_kinds="|".join(absent)))
    return sorted(complete), missing

In [ ]:
# Synthetic membership example; only counts are displayed.
example_catalog = {
    "annotation_person": {"DEMO_A_0": {}, "DEMO_B_0": {}},
    "labels": {"DEMO_A_0": {}, "DEMO_B_0": {}},
    "emr": {"DEMO_A_0": {}},
}
example_complete, example_missing = complete_records(example_catalog)
print("example complete records =", len(example_complete))  # 1
print("example incomplete records =", len(example_missing))  # 1

In [ ]:
complete_ids, missing_records = complete_records(catalog)
# Example: complete=[A, B, C], invalid={B} -> copy A and C only.
record_ids = [record_id for record_id in complete_ids if record_id not in invalid_records]
summary = pd.DataFrame([
    dict(kind=kind, discovered_files=found_counts[kind],
         unique_ids=len(catalog[kind]), planned_copies=len(record_ids))
    for kind in KINDS
])
display(summary)
print("복사할 기록 수:", len(record_ids))
print("종류가 빠진 기록 수:", len(missing_records))
print("오류가 있는 기록 수:", len(invalid_records))
# Issues in other records must not prevent copying the valid records.
can_copy = RUN is not None and bool(record_ids)
print("복사 가능:", can_copy)


## 5. 임시 폴더에 복사하고 해시를 확인한다

선택된 원본의 바이트를 그대로 복사한다. JSON을 재작성하거나 ID·임상값을 바꾸지 않는다. 출력 확장자만 `.json`으로 맞춰 00의 `glob("*.json")`이 읽게 한다.

복사한 파일의 SHA-256이 탐색 때의 값과 같은지 확인한다. 원본이 중간에 변경됐거나 파일 복사에 실패하면 그 기록의 임시 복사본 세 개를 모두 제거하고 다음 기록으로 넘어간다. 저장 공간 부족·임시 폴더 생성 또는 정리 실패는 복사를 중단하고 기존 `input_data`를 유지한다. 정상 복사한 기록이 하나도 없어도 기존 입력을 유지한다.


In [ ]:
def file_digest(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
def copy_record(catalog, record_id, stage):
    record_manifest = []
    for kind in KINDS:
        entry = catalog[kind][record_id]
        # Preserve the stem and raw bytes; normalize .JSON to .json for 00.
        relative = Path(kind) / f"{record_id}.json"
        destination = stage / relative
        shutil.copyfile(entry["path"], destination)
        if file_digest(destination) != entry["sha256"]:
            raise ValueError("copy_digest_mismatch")
        record_manifest.append(dict(
            kind=kind, record_id=record_id, source_path=entry["path"],
            destination=str(relative), sha256=entry["sha256"], size=entry["size"],
        ))
    # Only a fully verified triple becomes part of the final manifest.
    return record_manifest


In [ ]:
def stage_input(catalog, record_ids, stage, issues):
    stage.mkdir(exist_ok=False)
    for kind in KINDS:
        (stage / kind).mkdir()
    manifest = []
    for record_id in record_ids:
        try:
            record_manifest = copy_record(catalog, record_id, stage)
        except (OSError, ValueError) as error:
            # If the second JSON fails, remove the first as well: no partial triples.
            for kind in KINDS:
                (stage / kind / f"{record_id}.json").unlink(missing_ok=True)
            issues.append(dict(code="record_copy_failed", kind="", record_id=record_id,
                               path="", error_type=type(error).__name__))
            # A full/read-only destination affects every remaining record; stop staging.
            if isinstance(error, OSError) and error.errno in (errno.ENOSPC, errno.EDQUOT, errno.EROFS):
                raise
            continue
        manifest.extend(record_manifest)
    return manifest


In [ ]:
manifest = []
staged = False
if can_copy:
    try:
        manifest = stage_input(catalog, record_ids, RUN / "staged_input", issues)
        staged = bool(manifest)
    except (OSError, ValueError) as error:
        # Failure to create/clean the staging directory keeps the active input unchanged.
        issues.append(dict(code="staging_failed", kind="", record_id="", path="",
                           error_type=type(error).__name__))
print("복사 검증 완료:", staged)
print("검증한 파일 수:", len(manifest))


## 6. 내부 기록을 저장한 뒤 input_data를 교체한다

복사가 모두 검증되면 기존 `input_data`를 이번 실행 폴더의 `previous_input_data`로 옮겨 보관하고 새 폴더를 배치한다. 서로 다른 원본을 기존 입력에 덧붙이지 않으므로, 이전 기록이 섞이지 않는다. 새 폴더 배치에 실패하면 백업을 원위치로 되돌린다.

`inventory.csv`, `manifest.csv`, `missing_records.csv`, `issues.csv`에는 내부 추적용 경로·ID가 들어간다. 노트북에는 이 표를 출력하지 않는다. `status.json`에는 처리 상태와 집계만 남긴다.

In [ ]:
def save_audit(run, inventory, manifest, missing, issues):
    tables = {
        "inventory.csv": (inventory, ["kind", "record_id", "decision", "path", "sha256", "size"]),
        "manifest.csv": (manifest, ["kind", "record_id", "source_path", "destination", "sha256", "size"]),
        "missing_records.csv": (missing, ["record_id", "missing_kinds"]),
        "issues.csv": (issues, ["code", "kind", "record_id", "path", "error_type"]),
    }
    for filename, (rows, columns) in tables.items():
        pd.DataFrame(rows, columns=columns).to_csv(run / filename, index=False)

In [ ]:
def install_input(stage, target, backup):
    had_previous = target.exists()
    if had_previous:
        target.rename(backup)
    try:
        stage.rename(target)
    except OSError:
        # The old input remains usable if installing the new folder fails.
        if had_previous:
            backup.rename(target)
        raise
    return had_previous

In [ ]:
installed = False
previous_saved = False
if RUN is not None:
    try:
        # Persist the selected-source audit BEFORE changing the active input folder.
        save_audit(RUN, inventory, manifest, missing_records, issues)
        if staged:
            previous_saved = install_input(RUN / "staged_input", TARGET, RUN / "previous_input_data")
            installed = True
    except OSError as error:
        issues.append(dict(code="install_or_audit_failed", kind="", record_id="", path="",
                           error_type=type(error).__name__))
print("input_data 교체 완료:", installed)
print("기존 입력 백업:", previous_saved)

In [ ]:
state = "blocked"
if installed:
    state = "complete_with_issues" if issues or missing_records else "complete"
elif RUN is None:
    state = setup_code
elif not record_ids:
    state = "no_valid_records"
status = dict(
    status=state, ready_for_00=installed, records_copied=len(manifest) // 3 if installed else 0,
    files_copied=len(manifest) if installed else 0, incomplete_records=len(missing_records),
    invalid_records=len(invalid_records),
    copy_failed_records=sum(row["code"] == "record_copy_failed" for row in issues),
    duplicate_files=sum(row["decision"] != "first_copy" for row in inventory),
    skipped_links=scan_counts["skipped_links"], errors=len(issues), previous_input_saved=previous_saved,
)
if RUN is not None:
    try:
        save_audit(RUN, inventory, manifest, missing_records, issues)
        (RUN / "status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    except OSError:
        print("내부 실행 기록 저장 실패. 현재 input_data 교체 상태를먼저 확인하세요.")
print("상태:", state)
print("복사한 기록 수:", status["records_copied"])
print("오류 수:", status["errors"])


## 7. 다음 실행

`ready_for_00=True`이면 **00부터 차례로 다시 실행**한다. 이 노트북은 `data/prepared`나 이전 모델 결과를 갱신하지 않는다. 특히 09의 기존 CNN 체크포인트는 입력이 달라지면 재사용할 수 없다.

`complete_with_issues`는 일부 기록이나 폴더를 제외한 뒤 나머지 입력을 만든 상태다. `ready_for_00=True`이므로 00을 실행할 수 있지만 원본 전체를 가져온 것으로 해석하면 안 된다.

`blocked`·`no_valid_records` 또는 경로 오류 상태이면 이번 입력으로 교체하지 못한 것이다. 기존 입력이 남아 있다고 해서 새 원본 정리가 성공한 것으로 보지 않는다. 상세 원인은 내부 실행 폴더의 `issues.csv`·`missing_records.csv`에서 확인한다. 노트북에는 개별 파일 경로나 오류 전문을 표시하지 않는다.


In [ ]:
print("ready_for_00 =", status["ready_for_00"])
if installed:
    print("입력 저장 위치: 현재 작업 폴더의 input_data/")
    if state == "complete_with_issues":
        print("일부 기록이나 폴더를 제외했습니다. 제외 내역은 내부 실행 기록에 저장했습니다.")
    print("입력 정리 완료. 이제 00 → 01 → … → 09 순서로 실행하세요.")
else:
    print("이번 원본의 입력 정리는 완료되지 않았습니다. 00을 새 데이터 처리로 실행하지 마세요.")


### 반출용 오류 집계

파일 경로·ID 대신 코드에서 정한 오류 분류만 표시한다.

In [ ]:
# Codes are fixed by this notebook; no raw exception messages are displayed.
issue_codes = pd.Series([row["code"] for row in issues], dtype="string")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(issue_codes.value_counts().rename_axis("category").to_frame("count"))